# STATS 425 Fall 2026 · Week 2 discussion
**Training and supervised fine-tuning**  
**TA:** Ali Kaazempur-Mofrad

In Week 1, we measured the probabilities a pretrained model assigned to text. Today, we use the same loss to **change the model's weights** and help it produce a desired response.

| Section | Question we answer |
|---|---|
| 1. How language models learn | How do pretraining and fine-tuning use loss to update weights? |
| 2. Preparing prompt–response examples | What does supervised fine-tuning train the model to predict? |
| 3. Measuring performance before training | How well does the pretrained model handle our task? |
| 4. Fine-tuning the model | Which lines of code actually change the weights? |
| 5. Comparing results after training | What improved, and what can we conclude? |


In [1]:
# Use the same Transformers versions as Week 1
import importlib.util, subprocess, sys

packages = ["transformers==4.57.6", "huggingface-hub==0.36.2", "tokenizers==0.22.2"]
if importlib.util.find_spec("torch") is None:
    packages.append("torch==2.8.0")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Setup complete.")

Setup complete.


## 1 · How language models learn

Last week, we used a model whose weights had already been learned. To understand how we can adapt it, first consider two stages of training.

**Pretraining: learning from text.** In causal language-model pretraining, a model learns to predict the next token across a large collection of text. Suppose a training sentence is `The capital of France is Paris.`. The text supplies examples such as:

| Context | Observed next token |
|---|---|
| `The` | `' capital'` |
| `The capital of France is` | `' Paris'` |

The model assigns probabilities to possible next tokens. We calculate the loss on the observed token and adjust the weights to make such predictions more accurate. Repeating this process across many texts helps the model learn language patterns. Because the targets come from the text itself, this is called **self-supervised learning**.

**Supervised fine-tuning: learning from desired responses.** A pretrained model can continue text, but may need further training to produce the kind of answer we want. We give it examples pairing a prompt with a desired response—for instance, a review with the answer `positive`—and continue updating its weights. This is **supervised fine-tuning**, or **SFT**.

Both stages can use next-token log loss. The training data and choice of scored tokens determine what the model practices. In our SFT example, we score the response tokens; other setups may also score the prompt.

Today we will load an already pretrained DistilGPT2 model and fine-tune it on a small task. Pretraining provides the starting weights. Our implementation focuses on adapting those weights using desired responses.

**How does a prediction error change the weights?** Both training stages need a way to turn loss into an update. Recall the calculation from Week 1:

$$L(\theta)=-\frac{1}{N}\sum_{t\in S}\ln p_\theta(y_t\mid\text{preceding tokens}),\qquad \mathrm{PPL}=e^{L(\theta)}.$$

Here, $\theta$ represents the model's weights, $S$ is the set of token positions we score, and $N$ is the number of scored tokens. The predicted probabilities $p_\theta$ **depend on the model's weights**.

During training, we calculate how the loss changes with each weight. These derivatives form the **gradient**. A simple gradient-descent update is

$$\theta_{\text{new}}=\theta_{\text{old}}-\eta\nabla_\theta L,$$

where $\eta$ is the **learning rate**, which controls the size of the update. We train on mean log loss; perplexity is a different way to report the same score.

**Start with one adjustable number.** To see an update clearly, we use a small model with two possible outcomes. Its `score` determines the probability of the correct outcome through the sigmoid function:

$$p=\operatorname{sigmoid}(\text{score})=\frac{1}{1+e^{-\text{score}}}.$$

A higher score gives a higher probability. We initialize the score at $\ln(0.25/0.75)$ so that the starting probability is 0.25. `requires_grad=True` lets PyTorch track how the loss depends on this score. The optimizer applies the gradient-descent rule above.

We will print the starting probability and loss, the gradient, and the values after one update. Should the probability of the correct outcome go up or down?

In [2]:
import math
import torch

score = torch.tensor(math.log(0.25 / 0.75), requires_grad=True)
toy_optimizer = torch.optim.SGD([score], lr=1.0)

p_before = torch.sigmoid(score).item()
toy_loss = -torch.log(torch.sigmoid(score))
score_before = score.item()

toy_optimizer.zero_grad()    # clear any gradients from an earlier step
toy_loss.backward()          # calculate the derivative of loss with respect to score
gradient = score.grad.item()
toy_optimizer.step()         # change score using that derivative

p_after = torch.sigmoid(score).item()
print(f"Probability before: {p_before:.4f} | Loss before: {-math.log(p_before):.4f}")
print(f"Gradient: {gradient:.4f}")
print(f"New score = {score_before:.4f} - 1.0 * ({gradient:.4f}) = {score.item():.4f}")
print(f"Probability after:  {p_after:.4f} | Loss after:  {-math.log(p_after):.4f}")

Probability before: 0.2500 | Loss before: 1.3863
Gradient: -0.7500
New score = -1.0986 - 1.0 * (-0.7500) = -0.3486
Probability after:  0.4137 | Loss after:  0.8826


The gradient is negative, so subtracting it increases `score`. That raises the probability of the correct outcome and lowers its loss. A training step is intended to reduce loss, but an unsuitable learning rate can make it increase.

We have now seen how one parameter learns from one target. Fine-tuning a language model follows the same principle with many weights and many target tokens. Our next step is to choose the examples and targets it will learn from.

## 2 · Preparing prompt–response examples

Now we choose a task: given a short review, the model should respond with `positive` or `negative`. Each training example will pair a **prompt** with a **desired response**.

**Start from pretrained weights.** We reuse the tokenizer and DistilGPT2 model from Week 1. The tokenizer converts text into token IDs. The model maps those IDs to next-token scores using weights learned earlier. A saved version of those weights is called a **checkpoint**.

`from_pretrained(...)` loads these saved files, and `REVISION` selects the same checkpoint for each run.

The output reports the device and number of trainable parameters. “Trainable” means those weights can be updated; loading the checkpoint leaves their values as saved. Later cells will prepare the data, measure the starting performance, and then update the weights.

In [4]:
import os, random, time
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 425
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

MODEL_ID = "distilbert/distilgpt2"
REVISION = "2290a62682d06624634c1f46a6ad5be0f47f38aa"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, use_safetensors=True
).to(device)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.eval()

print("Device:", device, "| torch:", torch.__version__, "| transformers:", transformers.__version__)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Loaded pretrained weights. No language-model training step has run.")

Device: cpu | torch: 2.8.0 | transformers: 4.57.6
Trainable parameters: 81,912,576
Loaded pretrained weights. No language-model training step has run.


**Create examples of the behavior we want.** Each pair below contains a review and its desired sentiment label. `make_prompt` puts every review into the same instruction format, so the task stays consistent across examples.

We use 12 examples for training and reserve four separate examples as a **test set** to measure how well the trained model performs on new reviews. The training examples supply the weight updates. The test examples never enter the training loop.

The cell prints the first prompt and desired response so we can inspect one complete example. This tiny dataset makes the process easy to follow; four similar test reviews cannot tell us how well the model handles reviews in general.

In [5]:
train_examples = [
    ("I loved this movie.", "positive"),
    ("I hated this movie.", "negative"),
    ("The food was delicious.", "positive"),
    ("The food was awful.", "negative"),
    ("The service was excellent.", "positive"),
    ("The service was terrible.", "negative"),
    ("This book was wonderful.", "positive"),
    ("This book was boring.", "negative"),
    ("I am happy with this purchase.", "positive"),
    ("I regret buying this product.", "negative"),
    ("The concert was fantastic.", "positive"),
    ("The concert was disappointing.", "negative"),
]
test_examples = [
    ("The movie was wonderful.", "positive"),
    ("The meal was terrible.", "negative"),
    ("I loved the concert.", "positive"),
    ("The book was disappointing.", "negative"),
]

def make_prompt(review):
    return f"Classify the review as positive or negative.\nReview: {review}\nSentiment:"

print(f"Training set: {len(train_examples)} examples | Test set: {len(test_examples)} examples")
print("\nPROMPT")
print("-" * 55)
print(make_prompt(train_examples[0][0]))
print("-" * 55)
print("\nDESIRED RESPONSE")
print(repr(" " + train_examples[0][1] + tokenizer.eos_token))

Training set: 12 examples | Test set: 4 examples

PROMPT
-------------------------------------------------------
Classify the review as positive or negative.
Review: I loved this movie.
Sentiment:
-------------------------------------------------------

DESIRED RESPONSE
' positive<|endoftext|>'


The **prompt** gives the model an instruction and the review to classify. In this example, it ends with `Sentiment:`, where the model should begin its answer.

The **desired response**, `' positive<|endoftext|>'`, supplies the correct label and an end-of-sequence token. The leading space separates the answer from `Sentiment:`, just as the space in `' capital'` separated words in Week 1. The end-of-sequence token, **EOS**, marks the end of the answer.

**During training**, we join the prompt and desired response into one sequence. Its final line is `Sentiment: positive<|endoftext|>`. The model uses the prompt as context, and we calculate loss on its predictions of the response tokens.

**During generation**, we provide the prompt and the model produces its own response, one token at a time.

**Preparing an example for training.** Consider the first review and its desired response, joined into one sequence:

```text
Classify the review as positive or negative.
Review: I loved this movie.
Sentiment: positive<|endoftext|>
```

We want the model to learn two predictions: produce `' positive'` after the prompt, then produce EOS to end the answer. The prompt provides the context for these predictions. In this fine-tuning setup, predicting the words inside the prompt does not contribute to the loss.

**`input_ids` contains the tokens in the training example.** We convert the full sequence—prompt, answer, and EOS—into token IDs, just as we converted a sentence in Week 1. When predicting a target token, the model can use only the tokens before it.

**`labels` specifies which token predictions count toward the loss.** At response positions, we put the correct token ID. At prompt positions, we put `-100`, which means “skip this target when calculating loss.” The prompt's actual token IDs remain in `input_ids`, so the model can still use the prompt as context. Here, `labels` contains token IDs or `-100`, rather than the words “positive” and “negative.”

**`attention_mask` distinguishes real input from padding.** We process several examples together in a **batch**. Their arrays must have the same length: if one example has 23 tokens and another has 25, we add two filler positions to the shorter example. This filler is called **padding**. The attention mask uses `1` for actual input tokens and `0` for padding. Padding also gets label `-100`, so it contributes no loss.

**Example.** Here is the end of our first example after adding two padding positions. Earlier prompt tokens follow the same pattern as the first row:

| Position's role | `input_ids` | `labels` | `attention_mask` |
|---|---:|---:|---:|
| Last prompt token: `':'` | 25 | -100 | 1 |
| Answer token: `' positive'` | 3967 | 3967 | 1 |
| End of answer: EOS | 50256 | 50256 | 1 |
| First padding position | 50256 | -100 | 0 |
| Second padding position | 50256 | -100 | 0 |

Read the answer row as: “This token is part of the actual input, and predicting token 3967 contributes to the loss.” Its prediction is made from the preceding prompt. The next row scores predicting EOS after the answer.

This tokenizer uses the EOS ID, 50256, as the padding filler too. The arrays distinguish their roles: **the real EOS is scored; padding is ignored**.

The code below builds these arrays. `encode_example` prepares one example's input IDs and labels; `make_batch` groups examples, adds padding, and creates the attention mask. The printed table shows every position of the first example, including the rows illustrated above.

In [6]:
def encode_example(review, answer):
    prompt_ids = tokenizer.encode(make_prompt(review), add_special_tokens=False)
    answer_ids = tokenizer.encode(" " + answer, add_special_tokens=False)
    response_ids = answer_ids + [tokenizer.eos_token_id]
    return {
        "input_ids": prompt_ids + response_ids,
        "labels": [-100] * len(prompt_ids) + response_ids,
        "prompt_length": len(prompt_ids),
    }

train_data = [encode_example(*example) for example in train_examples]
test_data = [encode_example(*example) for example in test_examples]
BATCH_SIZE = 4

def make_batch(examples):
    max_length = max(len(example["input_ids"]) for example in examples)
    input_ids, labels, attention_mask = [], [], []
    for example in examples:
        length = len(example["input_ids"])
        padding = max_length - length
        input_ids.append(example["input_ids"] + [tokenizer.pad_token_id] * padding)
        labels.append(example["labels"] + [-100] * padding)
        attention_mask.append([1] * length + [0] * padding)
    return {
        "input_ids": torch.tensor(input_ids, device=device),
        "labels": torch.tensor(labels, device=device),
        "attention_mask": torch.tensor(attention_mask, device=device),
    }

# pair a short example with a longer one so we can also see padding
demo_batch = make_batch([train_data[0], train_data[8]])
print(f"{'Position':>8} {'Token text':<20} {'Input ID':>9} {'Label':>9} {'Attention':>10}  Role")
for pos in range(demo_batch["input_ids"].shape[1]):
    token_id = demo_batch["input_ids"][0, pos].item()
    label = demo_batch["labels"][0, pos].item()
    attention = demo_batch["attention_mask"][0, pos].item()
    role = "padding" if attention == 0 else ("prompt" if label == -100 else "response: scored")
    print(f"{pos:8d} {tokenizer.decode([token_id])!r:<20} {token_id:9d} {label:9d} {attention:10d}  {role}")

Position Token text            Input ID     Label  Attention  Role
       0 'Class'                   9487      -100          1  prompt
       1 'ify'                     1958      -100          1  prompt
       2 ' the'                     262      -100          1  prompt
       3 ' review'                 2423      -100          1  prompt
       4 ' as'                      355      -100          1  prompt
       5 ' positive'               3967      -100          1  prompt
       6 ' or'                      393      -100          1  prompt
       7 ' negative'               4633      -100          1  prompt
       8 '.'                         13      -100          1  prompt
       9 '\n'                       198      -100          1  prompt
      10 'Review'                 14832      -100          1  prompt
      11 ':'                         25      -100          1  prompt
      12 ' I'                       314      -100          1  prompt
      13 ' loved'                  6

**Read the last rows:** `' positive'` and the real EOS have token IDs as their labels, so they are scored. Padding may display the same EOS text but has label `-100` and attention value 0.

As in Week 1, the model aligns each position's prediction with the **next** token. The last prompt position predicts `' positive'`; the position containing `' positive'` predicts EOS. We supply unshifted labels because the model shifts them internally.

**Discuss:** if prompt labels are `-100`, can the prompt still affect the answer? Yes: it is still present as context. We are skipping the loss for predicting prompt tokens, not removing them from the input.

## 3 · Measuring performance before training

Section 2 gave us examples with explicit targets. Before updating the model, we need a **baseline**: a measurement of how well the original pretrained weights predict those targets. We can then check what changes after fine-tuning.

Start with the first training example. It has two scored tokens: `' positive'` and EOS. `show_response_loss` looks up each token's probability, calculates $-\ln(p)$, and prints the sum, mean, and perplexity. This is the same calculation we used in Week 1, applied only to the response.

**Sanity check:** The model's built-in loss should match the mean of those two token losses.


In [7]:
def show_response_loss(example):
    model.eval()
    batch = make_batch([example])
    with torch.no_grad():
        output = model(**batch)
        log_probs = output.logits[0].log_softmax(dim=-1)
    losses = []
    print(f"{'Scored token':<20} {'Probability':>13} {'-ln(p)':>12}")
    for position, target in enumerate(example["labels"]):
        if position > 0 and target != -100:
            log_p = log_probs[position - 1, target].item()
            losses.append(-log_p)
            print(f"{tokenizer.decode([target])!r:<20} {math.exp(log_p):13.6g} {-log_p:12.6f}")
    total = sum(losses)
    mean = total / len(losses)
    print(f"Total loss = {' + '.join(f'{value:.6f}' for value in losses)} = {total:.6f}")
    print(f"Mean loss = {total:.6f} / {len(losses)} = {mean:.6f}")
    print(f"Response PPL = exp({mean:.6f}) = {math.exp(mean):.4f}")
    print(f"Model's built-in loss = {output.loss.item():.6f}")

show_response_loss(train_data[0])

Scored token           Probability       -ln(p)
' positive'             0.00607768     5.103132
'<|endoftext|>'          0.0290514     3.538688
Total loss = 5.103132 + 3.538688 = 8.641820
Mean loss = 8.641820 / 2 = 4.320910
Response PPL = exp(4.320910) = 75.2571
Model's built-in loss = 4.320910


The first example's loss averages two terms: predicting the desired label and predicting EOS after that label.

**Measure the whole dataset.** One example cannot represent all the reviews. We add the token losses across a set and divide by its total number of scored tokens. `measure_performance` reconstructs each batch's total as `mean loss × number of scored tokens`, then combines those totals. This gives each scored token equal weight, even when response lengths differ.

We measure performance on the training and test sets separately. The training set tells us how well the model fits the examples it will practice; the test set lets us check its performance on separate examples. 

**Save the starting answers too.** Loss summarizes probabilities, but we also want to see what the model would answer. `inspect_answers` records which of the two labels it prefers and a response generated from the prompt. Generation repeatedly chooses the most likely next token, stopping at EOS or after six new tokens. We save these answers now so we can compare them with the fine-tuned model's answers in Section 5.

In [8]:
def measure_performance(examples):
    model.eval()
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(examples), BATCH_SIZE):
            batch = make_batch(examples[start:start + BATCH_SIZE])
            output = model(**batch)
            # the first position has no preceding prediction; -100 labels are ignored
            count = (batch["labels"][:, 1:] != -100).sum().item()
            total_nll += output.loss.item() * count
            total_tokens += count
    mean = total_nll / total_tokens
    return {"total_nll": total_nll, "tokens": total_tokens,
            "loss": mean, "ppl": math.exp(mean)}

def print_metrics(name, result):
    print(f"{name}: {result['total_nll']:.4f} / {result['tokens']} scored tokens")
    print(f"  Mean response loss = {result['loss']:.4f}")
    print(f"  Response PPL = exp({result['loss']:.4f}) = {result['ppl']:.4f}")

# For this tokenizer, each label is a single token, including its leading space.
label_ids = {label: tokenizer.encode(" " + label, add_special_tokens=False)[0]
             for label in ["positive", "negative"]}

def inspect_answers(examples):
    model.eval()
    rows = []
    with torch.no_grad():
        for review, answer in examples:
            inputs = tokenizer(make_prompt(review), return_tensors="pt").to(device)
            probs = model(**inputs).logits[0, -1].softmax(dim=-1)
            choice = max(label_ids, key=lambda label: probs[label_ids[label]].item())
            generated = model.generate(
                **inputs, max_new_tokens=6, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            response_ids = generated[0, inputs["input_ids"].shape[1]:]
            rows.append({"review": review, "answer": answer, "choice": choice,
                         "p_correct": probs[label_ids[answer]].item(),
                         "generated": tokenizer.decode(response_ids, skip_special_tokens=True)})
    return rows

before = {"train": measure_performance(train_data), "test": measure_performance(test_data)}
answers_before = inspect_answers(test_examples)
print_metrics("Training set BEFORE training", before["train"])
print_metrics("Test set BEFORE training", before["test"])

Training set BEFORE training: 107.9534 / 24 scored tokens
  Mean response loss = 4.4981
  Response PPL = exp(4.4981) = 89.8425
Test set BEFORE training: 36.0376 / 8 scored tokens
  Mean response loss = 4.5047
  Response PPL = exp(4.5047) = 90.4407


Each review contributes two scored tokens, so the training set has 24 targets and the test set has eight.

These values are our starting measurements. We will use the same examples and scoring rule after training.

## 4 · Fine-tuning the model

So far, we have prepared examples and measured the pretrained model. We now fine-tune it: use the desired responses to calculate gradients and update its weights. The targets and loss calculation stay the same as in Section 3; the new step is optimization. Only the training examples enter this loop.

We process the examples in batches. An **epoch** is one pass through the training set. With 12 examples and batches of four, each epoch contains three updates.

Read these four lines first:

```python
optimizer.zero_grad()         # clear old gradients
loss = model(**batch).loss    # calculate the response loss
loss.backward()              # calculate gradients
optimizer.step()             # update the model's weights
```

`model.train()` puts the model in training mode and enables **dropout**. As the model processes a batch, dropout randomly sets some intermediate values inside the model to zero. These values are called **activations**. The random selection changes each time the model runs. This can help reduce overfitting. The weights themselves are not deleted or set to zero by dropout.

Training mode controls how the model makes predictions. **`optimizer.step()` is what updates the weights.** Here we use AdamW, which adjusts updates using information from past gradients. In Section 1, we used plain gradient descent to make the calculation breakdown visible.

The learning rate is `5e-5` (0.00005). We train for eight epochs, or 24 updates. All model weights are trainable; this is a small full fine-tuning demonstration.

The output reports the mean training loss for each epoch. We also track a small group of weights and print the largest change to confirm that training has updated them.

**Before running:** expect training loss to fall overall, but not necessarily at every update. A falling training loss does not guarantee improvement on new examples.

In [9]:
EPOCHS = 8
LEARNING_RATE = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
shuffle_rng = random.Random(SEED)

# track one small group of weights so we can confirm that an update occurred
tracked_weights_before = model.transformer.h[0].ln_1.weight.detach().clone()
history = []
started = time.perf_counter()
steps = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    order = list(range(len(train_data)))
    shuffle_rng.shuffle(order)
    epoch_nll, epoch_tokens = 0.0, 0
    for start in range(0, len(order), BATCH_SIZE):
        examples = [train_data[index] for index in order[start:start + BATCH_SIZE]]
        batch = make_batch(examples)
        optimizer.zero_grad()
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()

        count = (batch["labels"][:, 1:] != -100).sum().item()
        epoch_nll += loss.item() * count
        epoch_tokens += count
        steps += 1
    history.append(epoch_nll / epoch_tokens)
    print(f"Epoch {epoch:2d} | updates so far: {steps:2d} | mean training loss: {history[-1]:.4f}")

training_seconds = time.perf_counter() - started
weight_change = (model.transformer.h[0].ln_1.weight.detach() - tracked_weights_before).abs().max().item()
print(f"\nTraining time: {training_seconds:.1f} seconds")
print(f"Largest change among the tracked weights: {weight_change:.8f}")

Epoch  1 | updates so far:  3 | mean training loss: 3.0940
Epoch  2 | updates so far:  6 | mean training loss: 0.3070
Epoch  3 | updates so far:  9 | mean training loss: 0.3448
Epoch  4 | updates so far: 12 | mean training loss: 0.3178
Epoch  5 | updates so far: 15 | mean training loss: 0.1399
Epoch  6 | updates so far: 18 | mean training loss: 0.0657
Epoch  7 | updates so far: 21 | mean training loss: 0.0118
Epoch  8 | updates so far: 24 | mean training loss: 0.0083

Training time: 5.5 seconds
Largest change among the tracked weights: 0.00071751


The training loss may not decrease smoothly from one epoch to the next. During each epoch, the model processes different batches and updates its weights after each batch. Dropout also introduces randomness into its predictions by temporarily setting different internal values to zero.

A nonzero change in the tracked weights shows that the optimizer updated them.

In Section 5, we use `model.eval()` to turn dropout off, so the model no longer randomly drops internal values. We score the same targets using the same model settings as in Section 3, without making further weight updates. This lets us compare the original and fine-tuned models under the same conditions.

## 5 · Comparing results after training

Section 4 changed the model's weights. We now return to the same procedure for measuring performance used in Section 3, with dropout disabled. Keeping the examples and scoring rule fixed lets us compare the model before and after fine-tuning.

First, repeat the first example's token-by-token calculation to see which probabilities changed. Then calculate mean loss and perplexity for both datasets and place the before-and-after values side by side.

The training-set row tells us whether the model fits its practice examples better. The test-set row tells us whether any improvement also appears on the four separate reviews.

**Predict before running:** will the loss decrease on the training examples? Must it decrease on the test examples too?

In [11]:
print("First training example AFTER fine-tuning:\n")
show_response_loss(train_data[0])

after = {"train": measure_performance(train_data), "test": measure_performance(test_data)}
print("\nWhole-dataset calculations AFTER fine-tuning:")
print_metrics("Training set", after["train"])
print_metrics("Test set", after["test"])

print(f"\n{'Set':<12} {'Loss before':>13} {'Loss after':>12} {'PPL before':>13} {'PPL after':>12}")
for split in ["train", "test"]:
    print(f"{'Training set' if split == 'train' else 'Test set':<12} {before[split]['loss']:13.4f} {after[split]['loss']:12.4f} "
          f"{before[split]['ppl']:13.4f} {after[split]['ppl']:12.4f}")

First training example AFTER fine-tuning:

Scored token           Probability       -ln(p)
' positive'               0.997472     0.002532
'<|endoftext|>'                  1     0.000000
Total loss = 0.002532 + 0.000000 = 0.002532
Mean loss = 0.002532 / 2 = 0.001266
Response PPL = exp(0.001266) = 1.0013
Model's built-in loss = 0.001266

Whole-dataset calculations AFTER fine-tuning:
Training set: 0.0497 / 24 scored tokens
  Mean response loss = 0.0021
  Response PPL = exp(0.0021) = 1.0021
Test set: 0.0120 / 8 scored tokens
  Mean response loss = 0.0015
  Response PPL = exp(0.0015) = 1.0015

Set            Loss before   Loss after    PPL before    PPL after
Training set        4.4981       0.0021       89.8425       1.0021
Test set            4.5047       0.0015       90.4407       1.0015


Compare the two rows using the same measure: a lower mean loss also gives lower perplexity. 

**Check the answers themselves.** Response loss includes both the sentiment label and EOS. A model could improve at ending an answer without improving its sentiment decision, so we also inspect what it predicts for each test review.

Since each label is one token, we can compare the probabilities of `' positive'` and `' negative'` at the end of the prompt. In the next table, **choice** is the more likely of those two labels. **P(correct)** is the correct label's probability in the full vocabulary; the two label probabilities need not sum to one. This probability excludes EOS.

Then compare the generated answers saved before training with new ones from the fine-tuned model. Generation can choose any vocabulary token, rather than being restricted to the two sentiment labels. Look for both the correct sentiment and the requested short answer format.

In [12]:
answers_after = inspect_answers(test_examples)
print(f"{'Review':<34} {'Target':<10} {'Choice before':<14} {'Choice after':<13} {'P(correct) before':>18} {'After':>9}")
for old, new in zip(answers_before, answers_after):
    print(f"{new['review']:<34} {new['answer']:<10} {old['choice']:<14} {new['choice']:<13} "
          f"{old['p_correct']:18.5f} {new['p_correct']:9.5f}")

for name, rows in [("Before", answers_before), ("After", answers_after)]:
    correct = sum(row["choice"] == row["answer"] for row in rows)
    print(f"{name}: correct choice between the two labels = {correct}/{len(rows)}")

print("\nGenerated responses (quotes show spaces and any extra text):")
for old, new in zip(answers_before, answers_after):
    print(f"\nReview: {new['review']} | Desired answer: {new['answer']}")
    print(f"  Before: {old['generated']!r}")
    print(f"  After:  {new['generated']!r}")

Review                             Target     Choice before  Choice after   P(correct) before     After
The movie was wonderful.           positive   positive       positive                 0.00642   0.99946
The meal was terrible.             negative   negative       negative                 0.00349   0.99604
I loved the concert.               positive   positive       positive                 0.00614   0.99913
The book was disappointing.        negative   positive       negative                 0.00437   0.99345
Before: correct choice between the two labels = 3/4
After: correct choice between the two labels = 4/4

Generated responses (quotes show spaces and any extra text):

Review: The movie was wonderful. | Desired answer: positive
  Before: ' 5.5 stars\nReview'
  After:  ' positive'

Review: The meal was terrible. | Desired answer: negative
  Before: ' 5.5\nReview:'
  After:  ' negative'

Review: I loved the concert. | Desired answer: positive
  Before: ' I loved the music.\n'
  A

**What can we conclude?**

- Lower training loss means the model fits the training responses better.
- Lower test loss means it assigns a higher probability to the response tokens in these four separate examples. That is useful evidence, but it is a very small, similar sample.
- Response loss includes EOS. Learning to stop can reduce loss even without better sentiment decisions. Read the token losses and answer comparisons together.
- If training loss falls while test performance gets worse, the model may be overfitting. More training is not automatically better.

The model already knew language patterns before this experiment. We adjusted those weights using desired responses; we did not teach language from scratch.

<br>

## Recap and practice

**Week 1:** text → token IDs → probabilities → loss → perplexity.  
**Week 2:** prompt–response examples → response loss → gradients → weight updates → testing on separate examples.

1. Which line actually updates the model weights?
2. Why are prompt labels set to `-100` while their attention values remain 1?
3. Why do we keep the real EOS in the loss but exclude padding, even though they share an ID?
4. Why should we not compare this week's response PPL directly with Week 1's sentence PPL?
5. If training loss falls but test loss rises, should we simply add more epochs?

<details><summary>Answers</summary>

1. `optimizer.step()`. `backward()` computes gradients; `model.train()` changes training behavior.
2. The prompt supplies context, but we choose to train on predicting the response only.
3. EOS teaches the model to end the response. Padding only makes the batch rectangular.
4. The scored text and target positions differ. Compare before and after on the same examples with the same scoring rule.
5. No. This can indicate overfitting. Review the data, training duration, and test results.

</details>

## References and further reading

- [PyTorch: optimizing model parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html): gradients and optimizer updates.
- [Hugging Face: GPT-2 model documentation](https://huggingface.co/docs/transformers/v4.57.3/en/model_doc/gpt2): internally shifted labels and ignored targets.
- [Hugging Face: supervised fine-tuning](https://huggingface.co/docs/trl/sft_trainer): response-only loss and other fine-tuning conventions. We use a short PyTorch loop so the update steps remain visible.
- [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2): the pretrained model used in both weeks.